In [1]:
import os
import sys
import re
import glob
from pprint import pprint
from pathlib import Path

import numpy as np
import pandas as pd

from reportlab.pdfgen import canvas
from pyteomics import pepxml, mzml

import matplotlib.pyplot as plt

import plotly.express as px
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import plotly.io as pio

import alphatims.bruker



In [ ]:
### Combine DIA data ###
### Sumarizes all stat files from the runs in the same folder into 1 tsv files for easier plotting showing ###

groups = [
    ["Probe 1-96_100SPD_diaPASEF_diann_res", "Probe 97-192_100SPD_diaPASEF_diann_res", "Probe 193-206_100SPD_diaPASEF_diann_res"],
    ["Probe 1-96_200SPD_diaPASEF_diann_res", "Probe 97-192_200SPD_diaPASEF_diann_res", "Probe 193-206_200SPD_diaPASEF_diann_res"],
    ["Probe 1-96_500SPD_diaPASEF_diann_res", "Probe 97-192_500SPD_diaPASEF_diann_res", "Probe 193-206_500SPD_diaPASEF_diann_res"],

]# DDA still needs to be added

group_names = ["100SPD_DIA", "200SPD_DIA", "500SPD_DIA"] 
path_to_descript_excel = "proteome_descript_199_updated.xlsx"
meta = pd.read_excel(path_to_descript_excel)

dfs = {}
for name, group in zip(group_names, groups):
    # Collect stats.tsv files in this group
    tsv_files = []
    for folder in group:
        #tsv_files.extend(glob.glob(os.path.normpath(os.path.join(folder, "*", "*stats.tsv"))))
        tsv_files.extend(Path(folder).glob("*/*stats.tsv"))

    # Read and merge into one DataFrame
    df = pd.concat(
        (
            pd.read_csv(f, sep="\t").assign(
                Group=name,
                Filename=f.name,
                Subfolder=f.parent.name,
                Root=f.parent.parent.name
            )
            for f in tsv_files
        ),
        ignore_index=True
    )

    pep_counts = []
    for f in tsv_files:
        report_path = f.parent / f"{f.parent.name}_report.tsv"
        if not report_path.exists():
            print("error")
            continue
        
        rep = pd.read_csv(report_path, sep="\t", usecols=["File.Name", "Stripped.Sequence"])
        
        peps = (
            rep.groupby("File.Name")["Stripped.Sequence"]
               .nunique()
               .reset_index(name="Peptides.Identified")
        )
        peps["Subfolder"] = f.parent.name
        pep_counts.append(peps)
    
    if pep_counts:
        pep_df = pd.concat(pep_counts, ignore_index=True)
        df = df.merge(pep_df, on=["File.Name", "Subfolder"], how="left")

    # Build a single regex from all 'Filename' keys (longest first) and case-insensitive
    keys = [k for k in meta["Filename"].unique().tolist() if k]
    pattern = "(?i)(" + "|".join(sorted(map(re.escape, keys), key=len, reverse=True)) + ")"

    df["Match"] = df["Filename"].astype(str).str.extract(pattern, expand=False)

    # Merge Proteins onto df via the extracted Match
    df = df.merge(
        meta.rename(columns={"Filename": "Match"}),
        on="Match",
        how="left"
    ).drop(columns=["Match"])
    dfs[name] = df

    # Save group-level merged DataFrame
    df.rename(columns={"Proteins": "Total.Protein.Number.Ref"}, inplace=True)
    df["Protein.Identification.Rate"] = (df["Proteins.Identified"] / df["Total.Protein.Number.Ref"]) * 100
    out_path = f"{name}_merged_stats.tsv"
    df.to_csv(out_path, sep="\t", index=False)
    print(f"Saved {out_path} with {len(df)} rows")

# Combine all runs
df_all = pd.concat(dfs.values(), ignore_index=True)

# Save combined file
out_all = "all_runs_merged_stats.tsv"
df_all.to_csv(out_all, sep="\t", index=False)
print(f"Saved {out_all} with {len(df_all)} rows")


In [ ]:
df_all = pd.read_csv("all_runs_merged_stats.tsv", sep="\t")

In [ ]:
import os
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import alphatims.bruker

d_folders = ["Probe 1-96_100SPD_ddaPASEF/", 
             "Probe 97-192_100SPD_ddaPASEF/",
             "Probe 193-206_100SPD_ddaPASEF/"]
result_folder = "100SPD_ddaPASEF_msfragger_philosopher_ionquant/"
sample_ids = os.listdir(result_folder)

output_tsv = "dda_100SPD/alphatims_metrics.tsv"

def qc_from_d(d_path: Path, sample_id: str) -> dict:
    """Extract MS1 TIC stats and MS2 precursor count from .d folder"""
    out = {"sample_id": sample_id}  # sample.d -> sample
    
    try:
        data = alphatims.bruker.TimsTOF(str(d_path))
    except Exception as e:
        out["error"] = f"open_failed: {e}"
        return out
    
    try:
        frames = data.frames
        ms1_frames = frames[frames["MsMsType"] == 0]
        ms2_frames = frames[frames["MsMsType"] == 8]
        
        # MS1 TIC median and CV
        if "SummedIntensities" in ms1_frames.columns and len(ms1_frames) > 0:
            tic = ms1_frames["SummedIntensities"].values.astype(float)
            tic = tic[np.isfinite(tic)]
            if len(tic) > 0:
                out["MS1_TIC_median"] = float(np.median(tic))
                if np.mean(tic) > 0:
                    out["MS1_TIC_CV"] = float(np.std(tic) / np.mean(tic))
                else:
                    out["MS1_TIC_CV"] = None
            else:
                out["MS1_TIC_median"] = None
                out["MS1_TIC_CV"] = None
        
        # MS2 precursor events 
        precursors = data.precursors
        precursors = precursors[precursors["Id"] > 0]
        out["MS2_precursor_events"] = int(len(precursors))
        
        out["MS1_frames"] = int(len(ms1_frames))
        out["MS2_frames"] = int(len(ms2_frames))
        out["Gradient_min"] = float(
            (frames["Time"].max() - frames["Time"].min()) / 60.0
        ) if len(frames) > 0 else None
        
    except Exception as e:
        out["error"] = f"processing_failed: {e}"
    
    return out

def main():
    all_rows = []

    for dda_folder in d_folders:
        for root, dirs, files in os.walk(dda_folder, topdown=True):
            for name in dirs:
                sample_id = name[:-2]
                if ".d" in name and sample_id in sample_ids:
                    d_folder_path = os.path.join(root, name)
                    if os.path.exists(d_folder_path):
                        print(d_folder_path)
                        row = qc_from_d(d_folder_path, sample_id)
                        all_rows.append(row)
                
        
    if not all_rows:
        print("No samples processed. Exiting.")
        sys.exit("Error")
    
    df = pd.DataFrame(all_rows)

    df.to_csv(output_tsv, sep="\t", index=False)
    print(f"\nWrote {len(df)} rows to {output_tsv}")

if __name__ == "__main__":
    main()

In [ ]:
import os
from pathlib import Path
import numpy as np
import pandas as pd

# Define your input folder
dda_folder = "100SPD_ddaPASEF_msfragger_philosopher_ionquant/"
output_tsv_fragpipe = "dda_100SPD/fragpipe_metrics.tsv"

def safe_read_tsv(filepath: str, **kwargs) -> pd.DataFrame | None:
    if not os.path.exists(filepath):
        return None
    try:
        return pd.read_csv(filepath, sep="\t", **kwargs, low_memory=False)
    except Exception as e:
        print(f"    failed to read {filepath}: {e}")
        return None

def qc_from_fragpipe_sample(sample_dir: Path) -> dict:
    out = {"sample_id": sample_dir.split("/")[-1]}
    
    # protein.tsv: identified proteins
    df_prot = safe_read_tsv(f"{sample_dir}/protein.tsv")
    if df_prot is not None:
        if "Is Decoy" in df_prot.columns:
            df_prot = df_prot[df_prot["Is Decoy"] == False]
        out["Identified_proteins"] = int(len(df_prot))
    else:
        out["Identified_proteins"] = None
    
    #  peptide.tsv: identified peptides  
    df_pep = safe_read_tsv(f"{sample_dir}/peptide.tsv")
    if df_pep is not None:
        out["Identified_peptides"] = int(len(df_pep))
    else:
        out["Identified_peptides"] = None
    
    #  psm.tsv: mass error, charge, missed cleavages 
    df_psm = safe_read_tsv(f"{sample_dir}/psm.tsv")
    if df_psm is not None:
        if "Is Decoy" in df_psm.columns:
            df_psm = df_psm[df_psm["Is Decoy"] == False].copy()
        else:
            df_psm = df_psm.copy()
        
        n_psm = len(df_psm)
        
        # Median precursor mass error in ppm
        if {"Delta Mass", "Calculated Peptide Mass"}.issubset(df_psm.columns):
            ppm = (df_psm["Delta Mass"] / df_psm["Calculated Peptide Mass"]) * 1e6
            ppm = ppm[np.isfinite(ppm)]
            if len(ppm):
                out["Median_precursor_mass_error_ppm"] = float(ppm.abs().median())
                out["Median_precursor_mass_error_ppm_signed"] = float(ppm.median())  # optional, for bias
            else:
                out["Median_precursor_mass_error_ppm"] = None
        
        # Charge state distribution 
        if "Charge" in df_psm.columns and n_psm > 0:
            charges = df_psm["Charge"].dropna()
            if len(charges) > 0:
                out["Pct_charge_2plus"] = float((charges == 2).mean() * 100)
                out["Pct_charge_3plus"] = float((charges == 3).mean() * 100)
            else:
                out["Pct_charge_2plus"] = None
                out["Pct_charge_3plus"] = None
        else:
            out["Pct_charge_2plus"] = None
            out["Pct_charge_3plus"] = None
        
        #  zero missed cleavages (its for the digestion efficiency)
        if "Number of Missed Cleavages" in df_psm.columns and n_psm > 0:
            zero = (df_psm["Number of Missed Cleavages"] == 0).sum()
            out["Pct_zero_missed_cleavages"] = float(zero / n_psm * 100)
        else:
            out["Pct_zero_missed_cleavages"] = None
    else:
        out["Median_precursor_mass_error_ppm"] = None
        out["Pct_charge_2plus"] = None
        out["Pct_charge_3plus"] = None
        out["Pct_zero_missed_cleavages"] = None
    
    #  ion.tsv: peak FWHM and ion mobility FWHM 
    df_ion = safe_read_tsv(f"{sample_dir}/ion.tsv")
    if df_ion is not None:
        # Filter decoys if present
        if "Is Decoy" in df_ion.columns:
            df_ion = df_ion[df_ion["Is Decoy"] == False]
        
        # Identified precursors
        out["Identified_precursors"] = int(len(df_ion))
        
        # median RT FWHM 
        if "Retention Time FWHM" in df_ion.columns:
            rt_fwhm = df_ion["Retention Time FWHM"].dropna()
            out["Median_RT_FWHM"] = float(rt_fwhm.median()) if len(rt_fwhm) else None
        else:
            out["Median_RT_FWHM"] = None
        
        # Ion mobility FWHM 
        if "Ion Mobility FWHM" in df_ion.columns:
            im_fwhm = df_ion["Ion Mobility FWHM"].dropna()
            out["Median_IM_FWHM"] = float(im_fwhm.median()) if len(im_fwhm) else None
        else:
            out["Median_IM_FWHM"] = None
    else:
        out["Identified_precursors"] = None
        out["Median_RT_FWHM"] = None
        out["Median_IM_FWHM"] = None
    
    return out

def main():
    all_rows = []
    sample_dirs = os.listdir(dda_folder)
    sample_dirs = [f"{dda_folder}{x}" for x in sample_dirs]

    for sample_path in sample_dirs:
        all_rows.append(qc_from_fragpipe_sample(sample_path))
        print(f"Processed Sample: {sample_path}")

        
    df = pd.DataFrame(all_rows)
    df.to_csv(output_tsv_fragpipe, sep="\t", index=False)
    print(f"\nWrote {len(df)} rows to {output_tsv_fragpipe}")


if __name__ == "__main__":
    main()

In [ ]:
df_d = pd.read_csv("dda_100SPD/alphatims_metrics.tsv", sep="\t")
df_fp = pd.read_csv("dda_100SPD/fragpipe_metrics.tsv", sep="\t")
df_proteome = pd.read_excel("proteome_descript_199_updated.xlsx", usecols="A, H")

sample_id_to_ref_prot = dict(zip(df_proteome['Filename'], df_proteome['Proteins']))
ref_prot = []

df_qc = df_fp.merge(df_d, on="sample_id", how="left", validate="one_to_one")

for dda_id in df_qc["sample_id"]:
    id = dda_id.split("_DDA100")[0]
    ref_prot.append(sample_id_to_ref_prot[id])

df_qc["Total.Protein.Number.Ref"] = ref_prot
df_qc["MS1_TIC_CV_pct"] = df_qc["MS1_TIC_CV"] * 100


df_qc["Protein.Identification.Rate"] = (df_qc["Identified_proteins"] / df_qc["Total.Protein.Number.Ref"]) * 100
print(df_qc)

df_qc.to_csv("dda_100SPD/qc_merged_100SPD_ddaPASEF.tsv", sep="\t", index=False)

In [ ]:
res_dia = Path(r"Path/to/subfolders/for/each/dia-spd-config/having the DIANN reports in there") # e.g. Folder/DIA100/*reports.tsv*
res_dda = Path(r"Path/to/subfolders/of/DDA/analysis/") # e.g. dda_results/sepcies1/*result files*
dia_dda_qc = Path(r"./Data/metadata_tables/new_combined/DIA_and_DDA_QC_metrics.xlsx")

df_qc_dia_100 = pd.read_excel(dia_dda_qc, sheet_name="DIA100")
df_qc_dia_200 = pd.read_excel(dia_dda_qc, sheet_name="DIA200")
df_qc_dia_500 = pd.read_excel(dia_dda_qc, sheet_name="DIA500")
#df_qc_dda_100 = pd.read_excel(dia_dda_qc, sheet_name="DDA100")

dict_dia_100_filename_ref_proteome_numb = (
    df_qc_dia_100
    .assign(Filename=df_qc_dia_100["Filename"].str.replace(".stats", "", regex=False))
    .set_index("Filename")["Total.Protein.Number.Ref"]
    .to_dict()
)
dict_dia_200_filename_ref_proteome_numb = (
    df_qc_dia_200
    .assign(Filename=df_qc_dia_200["Filename"].str.replace(".stats", "", regex=False))
    .set_index("Filename")["Total.Protein.Number.Ref"]
    .to_dict()
)
dict_dia_500_filename_ref_proteome_numb = (
    df_qc_dia_500
    .assign(Filename=df_qc_dia_500["Filename"].str.replace(".stats", "", regex=False))
    .set_index("Filename")["Total.Protein.Number.Ref"]
    .to_dict()
)
#dict_dda_100_filename_ref_proteome_numb = (
#    df_qc_dda_100
#    .assign(Filename=df_qc_dda_100["Filename"].str.replace(".stats", "", regex=False))
#    .set_index("Filename")["Total.Protein.Number.Ref"]
#    .to_dict()
#)

all_dicts = [
    dict_dia_100_filename_ref_proteome_numb,
    dict_dia_200_filename_ref_proteome_numb,
    dict_dia_500_filename_ref_proteome_numb,
#    dict_dda_100_filename_ref_proteome_numb,
]

# DIA
dia_proteins = set()
dia_peptides = set()


for path in Path(res_dia).rglob("*report*.tsv"):
    df = pd.read_csv(path, sep="\t",
                     usecols=["Stripped.Sequence", "Protein.Group", "Genes", "Global.Q.Value",
                              "Protein.Q.Value", "PG.Q.Value", "Global.PG.Q.Value",
                              "Q.Value", "Lib.Q.Value", "Lib.PG.Q.Value", "Proteotypic"])
    df_prot = df[(df["Protein.Q.Value"] <= 0.01) & (df["Proteotypic"] == 1)]
    df_pep = df[(df["Q.Value"] <= 0.01) & (df["Global.Q.Value"] <= 0.01)]

    df_abele = df.groupby("Protein.Group")["Stripped.Sequence"].nunique()
    print(len(set(df_abele[df_abele >= 2].index)))

    dia_proteins.update(df_prot["Genes"])
    dia_peptides.update(df_pep["Stripped.Sequence"])
    #print(len(set(df_pep["Stripped.Sequence"])))

    #dia_proteins.update(pep_per_group[pep_per_group >= 2].index)
    for d in all_dicts:
        if path.name in d:
            d[path.name] = len(set(df_abele[df_abele >= 2].index)) * 100 / d[path.name]
            if d[path.name] > 100:
                print(path.name, d[path.name])
                sys.exit("Stop at condition")

            break

print(f"The mean protein id rate for 100 DIA based on abele et al. (without MBR in this case) is: {np.mean(list(dict_dia_100_filename_ref_proteome_numb.values()))}")
print(f"The mean protein id rate for 200 DIA based on abele et al. (without MBR in this case) is: {np.mean(list(dict_dia_200_filename_ref_proteome_numb.values()))}")
print(f"The mean protein id rate for 500 DIA based on abele et al. (without MBR in this case) is: {np.mean(list(dict_dia_500_filename_ref_proteome_numb.values()))}")


n_dia_proteins = len(dia_proteins)
n_dia_peptides = len(dia_peptides)

# DDA analysis 
dda_peptides = set()
dda_proteins = set()

for path in Path(res_dda).rglob("peptide.tsv"):
    df = pd.read_csv(path, sep="\t", usecols=["Peptide"])
    dda_peptides.update(df["Peptide"])
    print(len(set(df["Peptide"])))

for path in Path(res_dda).rglob("protein.tsv"):
    df = pd.read_csv(path, sep="\t", usecols=["Gene"])
    dda_proteins.update(df["Gene"])

    print(len(set(df["Gene"])))

# Combined counts 
all_peptides = dia_peptides | dda_peptides
all_proteins = dia_proteins | dda_proteins

# Counts general
n_dia_peptides, n_dia_proteins = len(dia_peptides), len(dia_proteins)
n_dda_peptides, n_dda_proteins = len(dda_peptides), len(dda_proteins)
n_all_peptides, n_all_proteins = len(all_peptides), len(all_proteins)

print(f"DIA:      {n_dia_peptides} peptides, {n_dia_proteins} proteins")
print(f"DDA:      {n_dda_peptides} peptides, {n_dda_proteins} proteins")
print(f"Combined: {n_all_peptides} peptides, {n_all_proteins} proteins")
